# 👥 Employee Attrition Prediction — Machine Learning
### RISE Internship · Machine Learning & AI | Tamizhan Skills
---
**Project 4:** Industry-Oriented Machine Learning System for Employee Attrition Prediction  
**Tools:** Python · NumPy · Pandas · Matplotlib · Seaborn · Scikit-learn · Jupyter Notebook  
**Author:** _(Your Name)_ | RISE Internship 2026 | Gov. Polytechnic Nainital

---
### Problem Statement
Organizations face high costs due to employee attrition and struggle to identify factors
that cause employees to leave. Manual HR analysis is ineffective for large workforces.
Industry uses machine learning to **predict attrition** and improve **retention strategies**.

### Objective
Build a machine learning model that predicts employee attrition based on HR and performance data.

### Requirements Covered
✅ HR dataset ingestion · Data cleaning & preprocessing · Feature engineering for attrition factors  
✅ EDA · ML model training · Model evaluation (Accuracy + Recall)  
✅ Identification of key attrition drivers · Visualization of employee trends · Model documentation

In [ ]:
import subprocess, sys
for p in ['numpy','pandas','matplotlib','seaborn','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q','--break-system-packages'],stderr=subprocess.DEVNULL)
print('All libraries ready!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, pickle, os
from sklearn.model_selection  import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.svm              import SVC
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.metrics          import (accuracy_score, recall_score, precision_score,
                                      f1_score, classification_report,
                                      confusion_matrix, roc_auc_score, roc_curve)
plt.rcParams['figure.dpi']=110
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
print('Imports OK | NumPy', np.__version__, '| Pandas', pd.__version__)

## 1. HR Dataset Generation
Synthetic IBM-style dataset — 25 HR features, realistic attrition patterns (~17% rate).

> To use real data: `df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')`  
> IBM HR Analytics: [kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)

In [ ]:
def generate_hr_data(n=1000, seed=42):
    rng = np.random.default_rng(seed)
    DEPTS    = ['Human Resources','Research & Development','Sales']
    ROLES    = ['Sales Executive','Research Scientist','Laboratory Technician',
                'Manufacturing Director','Healthcare Representative',
                'Manager','Sales Representative','Research Director','Human Resources']
    dept     = rng.choice(DEPTS, n, p=[0.13,0.57,0.30])
    job_role = rng.choice(ROLES, n)
    gender   = rng.choice(['Male','Female'], n, p=[0.60,0.40])
    marital  = rng.choice(['Single','Married','Divorced'], n, p=[0.32,0.46,0.22])
    overtime = rng.choice(['Yes','No'], n, p=[0.28,0.72])
    edu_fld  = rng.choice(['Life Sciences','Medical','Marketing',
                           'Technical Degree','Human Resources','Other'],
                          n, p=[0.41,0.27,0.15,0.09,0.05,0.03])
    education= rng.integers(1,6,n)
    env_sat  = rng.integers(1,5,n)
    job_sat  = rng.integers(1,5,n)
    job_inv  = rng.integers(1,5,n)
    job_lvl  = rng.integers(1,6,n)
    rel_sat  = rng.integers(1,5,n)
    stock    = rng.integers(0,4,n)
    wlb      = rng.integers(1,5,n)
    perf     = rng.integers(3,5,n)
    age      = rng.integers(18,61,n)
    income   = np.round(rng.uniform(1000,20000,n),0).astype(int)
    num_co   = rng.integers(0,10,n)
    pct_hike = rng.integers(11,26,n)
    tot_yrs  = rng.integers(0,41,n)
    training = rng.integers(0,7,n)
    yrs_co   = rng.integers(0,41,n)
    yrs_role = np.clip(rng.integers(0,19,n),0,yrs_co)
    yrs_prom = np.clip(rng.integers(0,16,n),0,yrs_co)
    yrs_mgr  = np.clip(rng.integers(0,18,n),0,yrs_co)
    p = np.full(n,0.04)
    p += (overtime=='Yes')*0.20; p += (job_sat<=2)*0.14; p += (env_sat<=2)*0.09
    p += (wlb==1)*0.12; p += (yrs_co<3)*0.14; p += (income<3000)*0.10
    p += (marital=='Single')*0.08; p += (dept=='Sales')*0.05; p += (job_inv<=2)*0.07
    p += (stock==0)*0.05; p += (num_co>5)*0.05; p += (yrs_prom>6)*0.06
    p -= (yrs_co>15)*0.08; p -= (job_lvl>=3)*0.05; p -= (job_sat==4)*0.05
    p  = np.clip(p,0.01,0.95)
    attr = (rng.random(n)<p).astype(int)
    return pd.DataFrame({'EmployeeID':[f'EMP-{i+1:04d}' for i in range(n)],
        'Age':age,'Department':dept,'EducationField':edu_fld,'Education':education,
        'Gender':gender,'JobRole':job_role,'JobLevel':job_lvl,'MaritalStatus':marital,
        'OverTime':overtime,'EnvironmentSatisfaction':env_sat,'JobInvolvement':job_inv,
        'JobSatisfaction':job_sat,'RelationshipSatisfaction':rel_sat,'WorkLifeBalance':wlb,
        'PerformanceRating':perf,'StockOptionLevel':stock,'MonthlyIncome':income,
        'NumCompaniesWorked':num_co,'PercentSalaryHike':pct_hike,'TotalWorkingYears':tot_yrs,
        'TrainingTimesLastYear':training,'YearsAtCompany':yrs_co,'YearsInCurrentRole':yrs_role,
        'YearsSinceLastPromotion':yrs_prom,'YearsWithCurrManager':yrs_mgr,'Attrition':attr})

df = generate_hr_data(1000,42)
print(f'Shape: {df.shape} | Attrition rate: {df["Attrition"].mean()*100:.1f}%')
df.head()

## 2. Exploratory Data Analysis (EDA) & Employee Trends

In [ ]:
GREEN,RED,BLUE,GOLD,PURPLE='#22c55e','#ef4444','#3b82f6','#f59e0b','#7c3aed'
fig,axes=plt.subplots(1,2,figsize=(13,4))
sizes=[df['Attrition'].sum(),(df['Attrition']==0).sum()]
axes[0].pie(sizes,labels=['Left','Stayed'],colors=[RED,GREEN],autopct='%1.1f%%',
            startangle=90,textprops={'fontsize':12,'fontweight':'bold'},
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Attrition Distribution',fontsize=14,fontweight='bold')
ot=df.groupby('OverTime')['Attrition'].mean()*100
bars=axes[1].bar(ot.index,ot.values,color=[RED if v>20 else GREEN for v in ot.values],
                 edgecolor='white',width=0.4,alpha=0.9)
for bar,v in zip(bars,ot.values):
    axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,f'{v:.1f}%',
                 ha='center',fontsize=13,fontweight='bold')
axes[1].set_title('Attrition: OverTime vs No OverTime',fontsize=13,fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_ylim(0,max(ot.values)*1.3)
axes[1].spines[['top','right']].set_visible(False)
plt.suptitle('Attrition Overview',fontsize=15,fontweight='bold')
plt.tight_layout()
plt.savefig('attrition_overview.png',bbox_inches='tight',dpi=110)
plt.show()
print('KEY INSIGHT: OverTime employees churn at much higher rates!')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
da=df.groupby('Department')['Attrition'].mean()*100
c=[RED if v>20 else GOLD if v>14 else GREEN for v in da.values]
b1=axes[0].bar(da.index,da.values,color=c,edgecolor='white',width=0.5,alpha=0.9)
for bar,v in zip(b1,da.values):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f'{v:.1f}%',
                 ha='center',fontsize=11,fontweight='bold')
axes[0].set_title('Attrition by Department',fontsize=12,fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)');axes[0].set_xticklabels(da.index,rotation=15)
axes[0].set_ylim(0,max(da.values)*1.3);axes[0].spines[['top','right']].set_visible(False)
ra=df.groupby('JobRole')['Attrition'].mean().sort_values(ascending=True)*100
c2=[RED if v>25 else GOLD if v>15 else GREEN for v in ra.values]
axes[1].barh(ra.index,ra.values,color=c2,edgecolor='white',alpha=0.9)
axes[1].set_title('Attrition by Job Role',fontsize=12,fontweight='bold')
axes[1].set_xlabel('Attrition Rate (%)')
axes[1].spines[['top','right']].set_visible(False)
plt.suptitle('Department & Role Analysis',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('dept_role_analysis.png',bbox_inches='tight',dpi=110)
plt.show()

In [ ]:
sat_cols=['JobSatisfaction','EnvironmentSatisfaction','WorkLifeBalance','JobInvolvement']
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,col in zip(axes,sat_cols):
    s=df.groupby(col)['Attrition'].mean()*100
    c=[RED if v>20 else GOLD if v>12 else GREEN for v in s.values]
    ax.bar(s.index,s.values,color=c,edgecolor='white',width=0.6,alpha=0.9)
    ax.set_title(col.replace('Satisfaction','\nSatisfaction'),fontsize=9,fontweight='bold')
    ax.set_xlabel('Score (1=Low, 4=High)',fontsize=8)
    ax.set_ylabel('Attrition %')
    ax.set_ylim(0,max(s.values)*1.3)
    ax.spines[['top','right']].set_visible(False)
    for bar,v in zip(ax.patches,s.values):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f'{v:.0f}%',
                ha='center',fontsize=9,fontweight='bold')
plt.suptitle('Attrition by Satisfaction Scores (Visualization of Churn Factors)',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('satisfaction_analysis.png',bbox_inches='tight',dpi=110)
plt.show()
print('Employees with score 1 on any satisfaction dimension have highest attrition')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].hist(df[df['Attrition']==0]['YearsAtCompany'],bins=25,alpha=0.7,color=GREEN,label='Stayed',edgecolor='white')
axes[0].hist(df[df['Attrition']==1]['YearsAtCompany'],bins=25,alpha=0.7,color=RED,label='Left',edgecolor='white')
axes[0].set_title('Years at Company Distribution',fontsize=12,fontweight='bold')
axes[0].set_xlabel('Years');axes[0].set_ylabel('Count');axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)
axes[1].hist(df[df['Attrition']==0]['MonthlyIncome'],bins=30,alpha=0.7,color=GREEN,label='Stayed',edgecolor='white')
axes[1].hist(df[df['Attrition']==1]['MonthlyIncome'],bins=30,alpha=0.7,color=RED,label='Left',edgecolor='white')
axes[1].set_title('Monthly Income Distribution',fontsize=12,fontweight='bold')
axes[1].set_xlabel('Monthly Income ($)');axes[1].set_ylabel('Count');axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)
plt.suptitle('Tenure & Compensation Trends',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('employee_trends.png',bbox_inches='tight',dpi=110)
plt.show()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
data = df.drop(columns=['EmployeeID','Attrition']).copy()
y    = df['Attrition'].values
cat_cols = data.select_dtypes('object').columns.tolist()
num_cols = data.select_dtypes(exclude='object').columns.tolist()
print(f'Categorical: {cat_cols}')
print(f'Numerical: {num_cols}')
encoders={}
for col in cat_cols:
    le=LabelEncoder()
    data[col]=le.fit_transform(data[col])
    encoders[col]=le
scaler=StandardScaler()
data[num_cols]=scaler.fit_transform(data[num_cols])
X=data.values
feat_names=data.columns.tolist()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print(f'\nPreprocessing complete!')
print(f'Train: {X_train.shape} | Attrition rate: {y_train.mean()*100:.1f}%')
print(f'Test:  {X_test.shape}  | Attrition rate: {y_test.mean()*100:.1f}%')

## 4. Model Training

In [ ]:
models={'Logistic Regression':LogisticRegression(C=1.0,class_weight='balanced',random_state=42,max_iter=500),
        'Random Forest':RandomForestClassifier(n_estimators=100,class_weight='balanced',random_state=42,n_jobs=-1),
        'SVM (RBF)':SVC(kernel='rbf',C=1.0,class_weight='balanced',probability=True,random_state=42),
        'KNN':KNeighborsClassifier(n_neighbors=5)}
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
results={}
print(f"{'Model':<22}{'CV Recall':>10}{'CV Std':>8}{'Accuracy':>10}{'Recall':>8}{'ROC-AUC':>9}")
print('-'*70)
for name,model in models.items():
    cv=cross_val_score(model,X_train,y_train,cv=skf,scoring='recall')
    model.fit(X_train,y_train)
    yp=model.predict(X_test)
    ypr=model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else None
    results[name]={'model':model,'cv_recall':cv.mean(),'cv_std':cv.std(),
                   'accuracy':accuracy_score(y_test,yp),'recall':recall_score(y_test,yp),
                   'precision':precision_score(y_test,yp),'f1':f1_score(y_test,yp),
                   'roc_auc':roc_auc_score(y_test,ypr) if ypr is not None else None,
                   'y_pred':yp,'y_proba':ypr}
    r=results[name]
    auc=f"{r['roc_auc']*100:.2f}%" if r['roc_auc'] else 'N/A'
    print(f"{name:<22}{r['cv_recall']*100:>9.2f}%{r['cv_std']*100:>7.2f}%{r['accuracy']*100:>9.2f}%{r['recall']*100:>7.2f}%{auc:>9}")
print('-'*70)
best_name=max(results,key=lambda k:results[k]['recall'])
print(f'\nBest by Recall: {best_name} ({results[best_name]["recall"]*100:.2f}%)')

## 5. Model Evaluation — Accuracy & Recall

In [ ]:
best=results[best_name]
print(f'Classification Report — {best_name}')
print('='*55)
print(classification_report(y_test,best['y_pred'],target_names=['Stayed','Left']))

In [ ]:
n=len(results)
fig,axes=plt.subplots(1,n,figsize=(6*n,4))
if n==1: axes=[axes]
for ax,(name,res) in zip(axes,results.items()):
    cm=confusion_matrix(y_test,res['y_pred'])
    sns.heatmap(cm,annot=True,fmt='d',cmap='RdYlGn',ax=ax,
                xticklabels=['Stayed','Left'],yticklabels=['Stayed','Left'],
                linewidths=0.5,cbar=False,annot_kws={'size':13,'weight':'bold'})
    star=' ★' if name==best_name else ''
    ax.set_title(f'{name}{star}\nRecall:{res["recall"]*100:.1f}%  Acc:{res["accuracy"]*100:.1f}%',
                 fontweight='bold',fontsize=11,color='#4c1d95' if name==best_name else 'black')
    ax.set_xlabel('Predicted');ax.set_ylabel('Actual');ax.tick_params(axis='x',rotation=0)
plt.suptitle('Confusion Matrices',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png',bbox_inches='tight',dpi=110)
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
rc=['#3b82f6','#22c55e','#ef4444','#f59e0b']
for (name,res),col in zip(results.items(),rc):
    if res['y_proba'] is not None:
        fpr,tpr,_=roc_curve(y_test,res['y_proba'])
        ax.plot(fpr,tpr,color=col,linewidth=2.5 if name==best_name else 1.5,
                label=f"{name}  (AUC={res['roc_auc']:.3f})")
ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.4)
ax.set_xlabel('False Positive Rate');ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves — All Models',fontsize=13,fontweight='bold');ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('roc_curves.png',bbox_inches='tight',dpi=110)
plt.show()

## 6. Key Attrition Drivers (Feature Importance)

In [ ]:
rf=results['Random Forest']['model']
fi=pd.DataFrame({'Feature':feat_names,'Importance':rf.feature_importances_})
fi=fi.sort_values('Importance',ascending=True).tail(15)
clrs=[RED if v>fi['Importance'].quantile(0.75) else GOLD if v>fi['Importance'].quantile(0.5) else BLUE
      for v in fi['Importance']]
fig,ax=plt.subplots(figsize=(9,6))
ax.barh(fi['Feature'],fi['Importance'],color=clrs,edgecolor='white',alpha=0.9)
ax.set_title('Top 15 Attrition Drivers — Random Forest Feature Importance',fontsize=12,fontweight='bold')
ax.set_xlabel('Feature Importance')
legend=[mpatches.Patch(color=RED,label='High importance'),
        mpatches.Patch(color=GOLD,label='Medium importance'),
        mpatches.Patch(color=BLUE,label='Lower importance')]
ax.legend(handles=legend,fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('attrition_drivers.png',bbox_inches='tight',dpi=110)
plt.show()
print('Top 5 Attrition Drivers:')
print(fi.sort_values('Importance',ascending=False).head(5).to_string(index=False))

## 7. At-Risk Employee Identification

In [ ]:
best_model=results[best_name]['model']
all_proba=best_model.predict_proba(X)[:,1]
df_scored=df.copy()
df_scored['Attrition_Probability_%']=np.round(all_proba*100,1)
df_scored['Risk_Level']=pd.cut(all_proba,bins=[0,0.35,0.55,1.0],labels=['Low','Medium','High'])
at_risk=df_scored[all_proba>=0.55].sort_values('Attrition_Probability_%',ascending=False)
print(f'At-Risk Employees (≥55% probability): {len(at_risk)} ({len(at_risk)/len(df)*100:.1f}%)')
print(f'Actual attrition in at-risk group: {at_risk["Attrition"].sum()}')
cols=['EmployeeID','Department','JobRole','OverTime','JobSatisfaction',
      'YearsAtCompany','MonthlyIncome','Attrition_Probability_%','Risk_Level','Attrition']
print(at_risk[cols].head(10).to_string(index=False))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(all_proba,bins=40,color=PURPLE,alpha=0.75,edgecolor='white')
axes[0].axvline(0.55,color=RED,linewidth=2,linestyle='--',label='55% threshold')
axes[0].set_title('Attrition Probability Distribution',fontsize=12,fontweight='bold')
axes[0].set_xlabel('Attrition Probability');axes[0].set_ylabel('Employees')
axes[0].legend();axes[0].spines[['top','right']].set_visible(False)
rc=df_scored['Risk_Level'].value_counts().reindex(['High','Medium','Low']).fillna(0)
axes[1].pie(rc.values,labels=rc.index,colors=[RED,GOLD,GREEN],autopct='%1.1f%%',startangle=90,
            textprops={'fontsize':11,'fontweight':'bold'},wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Employee Risk Level Distribution',fontsize=12,fontweight='bold')
plt.suptitle('At-Risk Employee Analysis',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('risk_distribution.png',bbox_inches='tight',dpi=110)
plt.show()
at_risk[cols].to_csv('at_risk_employees.csv',index=False)
print('Saved: at_risk_employees.csv')

## 8. Save Model

In [ ]:
os.makedirs('saved_model',exist_ok=True)
bundle={'model':best_model,'scaler':scaler,'encoders':encoders,'features':feat_names}
with open('saved_model/attrition_model.pkl','wb') as f: pickle.dump(bundle,f)
print(f'Saved: saved_model/attrition_model.pkl')
print(f'Best model : {best_name}')
print(f'Test Accuracy: {results[best_name]["accuracy"]*100:.2f}%')
print(f'Test Recall  : {results[best_name]["recall"]*100:.2f}%')
print(f'ROC-AUC      : {results[best_name]["roc_auc"]*100:.2f}%')

## Summary

| Item | Detail |
|------|--------|
| Dataset | 1,000 synthetic IBM-style HR employees |
| Features | 25 (demographic + job + satisfaction + tenure) |
| Target | Attrition (0=Stayed, 1=Left) |
| Attrition Rate | ~17% |
| Models | Logistic Regression, Random Forest, SVM, KNN |
| Key Metrics | **Accuracy + Recall** (as per project requirements) |
| Key Deliverable | At-risk employee list + attrition driver analysis |

### Output Files
```
attrition_overview.png      Churn rate + overtime analysis
dept_role_analysis.png      Department & role attrition
satisfaction_analysis.png   Satisfaction scores vs attrition
employee_trends.png         Tenure & income distributions
confusion_matrices.png      All models comparison
roc_curves.png              ROC-AUC curves
attrition_drivers.png       Feature importance chart
risk_distribution.png       Risk level pie + probability histogram
at_risk_employees.csv       HR action list
saved_model/                Trained model + encoders + scaler
```

### Top HR Recommendations
1. **Overtime** — Audit workload; introduce comp time and flexible hours
2. **Low Job Satisfaction (score 1)** — Career counselling + role redesign
3. **Low Income (< $3K/month)** — Market benchmarking + salary adjustment
4. **New employees (< 3 years)** — Onboarding buddy + mentorship program
5. **No stock options** — ESOP / retention bonus for high-risk employees

---
*RISE Internship — Machine Learning & AI | Tamizhan Skills*